# Pyramidal HEALPix convolution -- single, small-region test

A simplified, fast companion to `pyramid_conv_sentinel2_test.ipynb`: instead
of running on the whole acquisition, this notebook crops a small region out
of one real product and builds a **fresh, small**
`HealPixDecomp`/`HealPixKernelPyramid`/`HealPixPyramidConv` scoped to that
region only -- cheap enough to iterate on quickly, no risk of the OOM issues
a full-scene run at deep `Jmax` can hit.

Two tests, run through the **same** pyramid (built once, in section 3):

- **A. Real data** (section 4): the cropped RGB region, before/after.
- **B. Single Dirac at the region's centre** (section 5): with everything
  else zero, the response is -- by construction -- this pipeline's point
  -spread function (PSF). Horizontal and vertical cuts through the centre
  are plotted against the analytic `KERNEL_SHAPE` profile it was built
  from, as a direct visual/numeric consistency check: does what the
  pipeline actually does to a point match what it was asked to do?

**Not executed here** -- same reason as `pyramid_conv_sentinel2_test.ipynb`:
this sandbox has no network access to `data.grid4earth.eu`. Run it where
that notebook already runs (e.g. Datarmor). If `healpix_analyse` changed on
disk since the kernel started, **restart the kernel** before re-running
(see that notebook's own note on this -- it applies here identically).

**Every stage below prints a diagnostic and raises immediately if its own
output is degenerate** (all-NaN, or effectively zero support) -- if a cell
raises, read its message first; it is telling you exactly which stage
produced nothing usable, instead of leaving it to be discovered several
cells later as a blank or saturated plot.


## 1. Parameters

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # to import g4e_source (next to this notebook)
sys.path.insert(0, str(Path.cwd().parent))    # to import healpix_analyse in dev mode

from g4e_source import G4E_L2A, G4E_PRODUCTS, RGB, ProductSeries

from healpix_analyse.decomp import HealPixDecomp
from healpix_analyse.kernel_pyramid import (
    HealPixKernelPyramid, kernel_gaussian, kernel_exponential, kernel_lorentzian, kernel_beta,
)
from healpix_analyse.pyramid_conv import HealPixPyramidConv
from healpix_analyse._ellipsoid import canonicalize_ellipsoid

# --------------------------------------------------------------------------
# Same convolution parameters as pyramid_conv_sentinel2_test.ipynb -- this
# notebook tests the *same* pipeline, just on a small crop instead of the
# whole scene.
# --------------------------------------------------------------------------
PRODUCTS    = list(G4E_PRODUCTS)
TIME_INDEX  = 0
DATA_LEVEL  = 17
SCALING     = "reflectance"
CACHE       = os.path.expanduser("~/s2_cache")

N_CHANNELS         = 3
JMAX               = 4     # fewer stages than the full-scene notebook (8): the
                            # crop below is small, a deep pyramid has nothing
                            # left to coarsen into past a few stages and just
                            # costs time -- raise it if SUB_N_CELLS (section 2)
                            # comes out large and you want to test a deeper PSF
COMPACT_KERNEL_SZ  = 5
KERNEL_SHAPE       = kernel_gaussian
SIGMA_PIX          = 1.2
GAUGE_TYPE         = "phi"

# Plain angular radius (not a fraction of the scene's own bounding box --
# see the module-docstring note above on why that was fragile). The centre
# is picked automatically in section 2, from the data itself.
CROP_RADIUS_DEG = 0.4   # degrees; raise this if section 2 reports too few/too
                         # degraded cells, or if section 5's "support at
                         # centre" ends up well below the domain max (crop
                         # too tight for this JMAX -- see section 5's check)

DTYPE  = torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 2. Reading a real acquisition and cropping a small region

Same read as `pyramid_conv_sentinel2_test.ipynb` section 2 (whole scene,
level 17 is already the coarsest published level, so this stays cheap).

The crop centre is the **circular-safe centroid of the cells that actually
have finite data** for `TIME_INDEX` (not the geometric centre of the whole
footprint's bounding box, which can sit in a gap the swath never covers,
and which breaks silently if the footprint straddles the antimeridian --
see the module docstring). Cells are then selected by plain angular
(great-circle) distance from that centre, which is rotation- and
wraparound-safe by construction -- no axis-aligned lon/lat box involved.


In [ ]:
src = ProductSeries(PRODUCTS, level=DATA_LEVEL, base=G4E_L2A, bands=RGB, scaling=SCALING)
level, cell_id, dates = src.level, src.cell_id, src.dates
print(f"HEALPix level {level}, {cell_id.size} cells, product {dates[TIME_INDEX]}")

rgb = src.rgb(TIME_INDEX, cache=CACHE)
finite = np.isfinite(rgb).all(axis=1)
print(f"finite (non-NaN, all 3 bands) in full scene: {100 * finite.mean():.1f}% of {rgb.shape[0]} cells")
if not finite.any():
    raise ValueError(
        f"product {dates[TIME_INDEX]} ({TIME_INDEX=}) has NO finite pixel anywhere in the scene -- "
        "try a different TIME_INDEX"
    )

import healpix_geo
lon_deg, lat_deg = healpix_geo.nested.healpix_to_lonlat(
    cell_id.tolist(), level, ellipsoid=canonicalize_ellipsoid(src.ellipsoid),
)
lon_deg, lat_deg = np.asarray(lon_deg), np.asarray(lat_deg)


def haversine_deg(lon1, lat1, lon2, lat2):
    """Angular great-circle distance (degrees) between (lon1,lat1) and (lon2,lat2), degrees in.

    Used throughout this notebook instead of a raw lon/lat difference,
    which is wrong near the poles and breaks across the antimeridian.
    """
    lon1r, lat1r, lon2r, lat2r = np.radians(lon1), np.radians(lat1), np.radians(lon2), np.radians(lat2)
    dlon, dlat = lon2r - lon1r, lat2r - lat1r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return np.degrees(2 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0))))


# Circular-safe centroid of the FINITE cells only: averaging unit vectors
# (not raw degrees) means a footprint straddling lon=+/-180 still gives a
# sensible centre, unlike 0.5*(lon.min()+lon.max()).
lon_rad, lat_rad = np.radians(lon_deg[finite]), np.radians(lat_deg[finite])
lon0 = float(np.degrees(np.arctan2(np.mean(np.sin(lon_rad)), np.mean(np.cos(lon_rad)))))
lat0 = float(np.mean(lat_deg[finite]))

theta_deg = haversine_deg(lon_deg, lat_deg, lon0, lat0)
in_crop = theta_deg < CROP_RADIUS_DEG

sub_cell_id = cell_id[in_crop]
sub_lon = lon_deg[in_crop]
sub_lat = lat_deg[in_crop]
sub_rgb = rgb[in_crop]
SUB_N_CELLS = sub_cell_id.size
sub_finite_frac = float(np.isfinite(sub_rgb).all(axis=1).mean()) if SUB_N_CELLS else 0.0

print(f"crop: radius {CROP_RADIUS_DEG} deg around (lon={lon0:.3f}, lat={lat0:.3f}), "
      f"{SUB_N_CELLS} cells, {100 * sub_finite_frac:.1f}% finite")
if SUB_N_CELLS < 200 or sub_finite_frac < 0.3:
    raise ValueError(
        f"crop too small or too degraded ({SUB_N_CELLS} cells, {100 * sub_finite_frac:.1f}% finite) -- "
        "raise CROP_RADIUS_DEG in section 1, or try a different TIME_INDEX "
        "(this date may still be mostly cloudy right at its own valid-data centroid)"
    )
finite_sub_vals = sub_rgb[np.isfinite(sub_rgb)]
print(f"reflectance range in the crop (finite pixels): min={finite_sub_vals.min():.4g}, "
      f"max={finite_sub_vals.max():.4g}, mean={finite_sub_vals.mean():.4g}")

# nearest actual cell to the crop's own centre -- this is where the Dirac in
# section 5 gets placed, and what every cross-section below is centred on
center_idx = int(np.argmin(haversine_deg(sub_lon, sub_lat, lon0, lat0)))
center_lon, center_lat = float(sub_lon[center_idx]), float(sub_lat[center_idx])
print(f"centre cell: index {center_idx} in the crop, at (lon={center_lon:.4f}, lat={center_lat:.4f})")


## 3. A fresh, small pyramid scoped to the crop

Built once, reused identically for both tests below (real data in section
4, Dirac in section 5) -- this is the actual point of the notebook: the
same pipeline is exercised on two different inputs so its response to a
point (section 5) can be read as a diagnostic for what it does to real data
(section 4), not as a separate, unrelated experiment.


In [ ]:
decomp = HealPixDecomp(
    level=level, cell_ids=sub_cell_id, Jmax=JMAX,
    ellipsoid=src.ellipsoid, dtype=DTYPE, device=DEVICE,
)
print(decomp)
print("band sizes (fine -> coarse):", decomp.sizes)
if any(s == 0 for s in decomp.sizes):
    raise ValueError(
        f"a pyramid band is empty ({decomp.sizes}) -- JMAX={JMAX} is too deep for a crop of only "
        f"{SUB_N_CELLS} cells; lower JMAX in section 1 (or raise CROP_RADIUS_DEG)"
    )

kernel_pyramid = HealPixKernelPyramid.from_kernel(
    decomp, KERNEL_SHAPE(sigma_pix=SIGMA_PIX),
    compact_kernel_sz=COMPACT_KERNEL_SZ, gauge_type=GAUGE_TYPE,
    channels=N_CHANNELS, ellipsoid=src.ellipsoid, dtype=DTYPE,
)
pconv = HealPixPyramidConv(decomp, kernel_pyramid, mode="normalized")

import healpix_plot
grid_plot = healpix_plot.HealpixGrid(
    level=level, indexing_scheme="nested", ellipsoid=canonicalize_ellipsoid(src.ellipsoid),
)
# Built from the crop's own centre + radius (not sub_lon.min()/max()) --
# a raw min/max is wrong whenever the crop straddles the antimeridian, the
# same pitfall the crop selection itself avoids in section 2.
view = (lon0 - CROP_RADIUS_DEG, lon0 + CROP_RADIUS_DEG,
        max(lat0 - CROP_RADIUS_DEG, -90.0), min(lat0 + CROP_RADIUS_DEG, 90.0))

# the finest band's own angular pixel spacing, in radians -- the exact
# convention KERNEL_SHAPE(rho_pix, phi) is defined against (see the
# KernelFn docstring in healpix_analyse/kernel_pyramid.py); reused in
# section 5 to convert measured angular offsets into the same pixel units.
nside = 2 ** level
alpha_pix = np.sqrt(4.0 * np.pi / (12.0 * nside ** 2))


## 4. Test A -- real cropped data, before / after

In [ ]:
import cartopy.crs as ccrs

x_real = torch.as_tensor(sub_rgb.T, dtype=DTYPE, device=DEVICE)
with torch.no_grad():
    y_real, support_real = pconv(x_real, return_support=True)
rgb_filtered = y_real.detach().cpu().numpy().T
support_map = support_real.detach().cpu().numpy()[0]

print(f"support: min {np.nanmin(support_map):.4g}, max {np.nanmax(support_map):.4g}, "
      f"has NaN {np.isnan(support_map).any()}")
if not np.isfinite(support_map).any() or np.nanmax(support_map) <= 0:
    raise RuntimeError(
        "support is entirely zero/NaN over the whole crop -- the pyramid produced no usable output "
        "at all here; re-check the band sizes printed in section 3"
    )

hi = float(np.nanpercentile(sub_rgb, 98))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5),
                          subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
healpix_plot.plot(sub_cell_id, sub_rgb, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                   view=view, ax=axes[0], rgb_clip=(0.0, hi), axis_labels="none", title="before (real)")
healpix_plot.plot(sub_cell_id, rgb_filtered, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                   view=view, ax=axes[1], rgb_clip=(0.0, hi), axis_labels="none",
                   title=f"after (Jmax={JMAX}, {COMPACT_KERNEL_SZ}x{COMPACT_KERNEL_SZ})")
mp = healpix_plot.plot(sub_cell_id, support_map, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                        view=view, ax=axes[2], axis_labels="none", title="synthesized support",
                        vmin=0.0, vmax=float(np.nanmax(support_map)))
fig.colorbar(mp, ax=axes[2], shrink=0.7)
plt.show()


## 5. Test B -- single Dirac at the centre: the pipeline's own PSF

Everything zero except the centre cell (all 3 channels set to 1) --
deliberately **not** derived from `sub_rgb`, so this test does not depend
on whether the real data happens to be cloudy anywhere in the crop. The
output is, by construction, this exact pipeline's point-spread function --
what it actually does to a point, as opposed to what `KERNEL_SHAPE`/
`SIGMA_PIX` asked it to do.


In [ ]:
rgb_dirac = np.zeros_like(sub_rgb, dtype=np.float32)
rgb_dirac[center_idx, :] = 1.0

x_dirac = torch.as_tensor(rgb_dirac.T, dtype=DTYPE, device=DEVICE)
with torch.no_grad():
    y_dirac, support_dirac = pconv(x_dirac, return_support=True)
rgb_psf = y_dirac.detach().cpu().numpy().T          # [SUB_N_CELLS, 3]
support_psf = support_dirac.detach().cpu().numpy()[0]

psf = rgb_psf[:, 0]   # the 3 channels are identical (block-diagonal, same profile -- see class docstring)
print(f"PSF: min {psf.min():.4g}, max {psf.max():.4g}, has NaN {np.isnan(psf).any()}, "
      f"finite fraction {np.isfinite(psf).mean():.3f}")
print(f"support at centre: {support_psf[center_idx]:.4g} (domain max: {support_psf.max():.4g})")
if not np.isfinite(psf).any() or np.abs(psf).max() == 0:
    raise RuntimeError(
        "PSF is entirely NaN/zero over the whole crop -- a centred Dirac produced no usable response "
        "at all, independently of the real data (this input has no NaN of its own). Check the band "
        "sizes printed in section 3 -- an empty or size-1 band would explain this."
    )
if support_psf[center_idx] < 0.5 * support_psf.max():
    print(
        "WARNING: support at the centre is well below the domain max -- the crop is probably too "
        "small for this JMAX (the crop's own edge is contaminating the centre); raise "
        "CROP_RADIUS_DEG in section 1 and re-run before trusting the cuts below."
    )


In [ ]:
# Resample the PSF onto a small regular lon/lat grid (nearest, same
# convention healpix_plot.plot uses internally). Also used to pick a tight
# display window around the actual response for section 5's 2D map --
# SIGMA_PIX is typically only a few pixels wide, easy to miss entirely in a
# map drawn at the crop's own (much larger) extent.
from healpix_plot.resampling import resample as hp_resample

XSECT_SHAPE = 65   # odd, so there is an exact centre row/column
target_grid, psf_image = hp_resample(
    sub_cell_id, psf, sampling_grid={"shape": XSECT_SHAPE}, healpix_grid=grid_plot,
    interpolation="nearest", agg="mean",
)
xs = target_grid.x[0, :]   # longitudes, degrees
ys = target_grid.y[:, 0]   # latitudes, degrees
row_idx = int(np.argmin(np.abs(ys - center_lat)))   # horizontal cut: fixed latitude
col_idx = int(np.argmin(np.abs(xs - center_lon)))   # vertical cut:   fixed longitude

# tight zoom window: the smallest box (padded 50%) containing every sample
# above 2% of the PSF's own peak, so the response is actually visible
peak = np.nanmax(np.abs(psf_image))
significant = np.isfinite(psf_image) & (np.abs(psf_image) > 0.02 * peak)
if significant.any():
    lon_lo, lon_hi = xs[np.where(significant.any(axis=0))[0][[0, -1]]]
    lat_lo, lat_hi = ys[np.where(significant.any(axis=1))[0][[0, -1]]]
    pad_lon = 0.5 * max(lon_hi - lon_lo, 4 * (xs[1] - xs[0]))
    pad_lat = 0.5 * max(lat_hi - lat_lo, 4 * (ys[1] - ys[0]))
    zoom_view = (lon_lo - pad_lon, lon_hi + pad_lon, lat_lo - pad_lat, lat_hi + pad_lat)
else:
    zoom_view = view   # fell back to the full crop -- nothing rose above 2% of peak anywhere

fig, ax = plt.subplots(figsize=(5, 4.3), subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
mp = healpix_plot.plot(sub_cell_id, psf, healpix_grid=grid_plot, sampling_grid={"shape": 300},
                        view=zoom_view, ax=ax, axis_labels="none",
                        vmin=0.0, vmax=float(peak), title=f"PSF (response to a centred Dirac, Jmax={JMAX})")
fig.colorbar(mp, ax=ax, shrink=0.8)
plt.show()


In [ ]:
h_row_lat = ys[row_idx]
theta_h = haversine_deg(xs, h_row_lat, center_lon, h_row_lat)
rho_h = np.sign(xs - center_lon) * np.radians(theta_h) / alpha_pix
cut_h = psf_image[row_idx, :]

v_col_lon = xs[col_idx]
theta_v = haversine_deg(v_col_lon, ys, v_col_lon, center_lat)
rho_v = np.sign(ys - center_lat) * np.radians(theta_v) / alpha_pix
cut_v = psf_image[:, col_idx]

kernel_fn = KERNEL_SHAPE(sigma_pix=SIGMA_PIX)
rho_theory = np.linspace(min(rho_h.min(), rho_v.min()), max(rho_h.max(), rho_v.max()), 400)
theory = kernel_fn(np.abs(rho_theory), np.zeros_like(rho_theory))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, rho, cut, label in (
    (axes[0], rho_h, cut_h, "horizontal cut (varying longitude, latitude fixed)"),
    (axes[1], rho_v, cut_v, "vertical cut (varying latitude, longitude fixed)"),
):
    finite_cut = np.isfinite(cut)
    print(f"{label}: {finite_cut.sum()}/{finite_cut.size} finite samples")
    if not finite_cut.any():
        print("  -> nothing to plot for the measured curve (all NaN); only the analytic curve will show")
    cut_peak = np.nanmax(np.abs(cut[finite_cut])) if finite_cut.any() else 1.0
    ax.plot(rho[finite_cut], cut[finite_cut] / max(cut_peak, 1e-12), "o-", ms=3, label="measured PSF (peak-normalized)")
    ax.plot(rho_theory, theory / theory.max(), "--", label=f"KERNEL_SHAPE (sigma_pix={SIGMA_PIX})")
    ax.axvline(0, color="0.7", lw=0.8)
    ax.set_xlabel("signed distance from centre (pixel units, finest band)")
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
plt.show()


## Reading the result

- If the measured PSF (solid) tracks the dashed analytic curve closely near
  the centre, the pipeline reconstructs the requested kernel shape well at
  this `JMAX`/`SIGMA_PIX`/`COMPACT_KERNEL_SZ`.
- A measured curve **wider** than the analytic one, or with visible
  shoulders/ringing the analytic curve doesn't have, is exactly what the
  per-band, block-diagonal construction in `from_kernel` (as opposed to the
  jointly-fit `calibrate_joint`, see `docs/pyramid_convolution.md` section
  D.1) is expected to add on top of the requested profile once `Jmax > 0`
  -- each pyramid band reconstructs its own share of the response
  independently, so their sum is not guaranteed to reduce back to the
  single analytic kernel evaluated at the finest resolution. This section
  is exactly how to *see* that gap on a specific `SIGMA_PIX`/`JMAX`
  combination, rather than only reading about it in the docs.
- If the printed "finite samples" count above was 0 for either cut, or any
  cell above raised, **read that cell's own printed message first** -- it
  names exactly which stage produced nothing usable, rather than guessing
  from a blank plot.
